# Task 3: Local 7B Parameter LLM Quantization & Logit Extraction Pipeline

## Objective

To deploy a locally quantized Llama-3-8B-Instruct model and analyze its hidden layers, token logits, attention weights, and entropy during text generation.

## Technologies / Tools Used

- Ollama
- Llama 3-8B-Instruct (GGUF)
- PyTorch
- Hugging Face Transformers
- Google Colab

## Formula

### Quantization

\[
W_q = \text{Quantize}(W)
\]

Quantization reduces model memory requirements by representing model weights using lower-precision values.

### Entropy

\[
H(P)=-\sum_i P_i\log(P_i)
\]

Entropy measures the uncertainty of the model's predicted token distribution.

In [4]:
# Install required system packages and Ollama

!apt-get update -qq
!apt-get install -y zstd -qq
!curl -fsSL https://ollama.com/install.sh | sh

# Install Python libraries

!pip -q install torch transformers requests

print("Installation completed.")
!ollama --version

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package zstd.
(Reading database ... 118243 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
Installation completed.


## Step 1: Start Ollama Server

Start the Ollama service so that the local language model can be accessed from the notebook.

In [5]:
import subprocess
import time

ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

time.sleep(5)

print("Ollama server started.")

Ollama server started.


## Step 2: Download and Verify the Local Llama Model

Download the Llama 3 8B model using Ollama and verify that it is available locally.

In [6]:
# Download Llama 3 8B

!ollama pull llama3:8b

# Display installed models

!ollama list


NAME         ID              SIZE      MODIFIED               
llama3:8b    365c0bd3c000    4.7 GB    Less than a second ago    


## Step 3: Generate Text from the Local Model

Send a prompt to the locally running Llama 3 model and display its response.

In [7]:
import subprocess

prompt = "Explain generative artificial intelligence in simple words."

result = subprocess.run(
    ["ollama", "run", "llama3:8b", prompt],
    capture_output=True,
    text=True
)

print("Generated Response:")
print(result.stdout)

Generated Response:
Generative Artificial Intelligence (AI) is a type of AI that creates new, o
original content, such as:

1. Images: Like painting or drawing.
2. Music: Like composing music.
3. Text: Like writing stories or articles.
4. Videos: Like making short films.

These AIs use algorithms to generate this content based on what they've lea
learned from a dataset (a collection of examples) provided by humans. The m
more data they have, the better they can create new, unique content that re
resembles what they've seen before.

Think of it like a child playing with building blocks. They start with what
what they know (the blocks) and use their imagination to create something n
new and original (like a castle or a robot). Generative AI does the same th
thing, but with data instead of blocks!

Some examples of generative AI in action:

* You can ask a chatbot like me to generate a short story or poem based on 
a prompt.
* Music streaming services use generative AI to suggest new song

## Step 4: Extract Token Probabilities and Calculate Entropy

Use the Ollama API to generate tokens while requesting probability information. Entropy is calculated from the token probability distribution to measure uncertainty during generation.

In [8]:
import requests
import json
import math

url = "http://localhost:11434/api/generate"

payload = {
    "model": "llama3:8b",
    "prompt": "What is artificial intelligence?",
    "stream": False,
    "options": {
        "temperature": 0
    }
}

response = requests.post(
    url,
    json=payload
)

data = response.json()

print("Generated Text:")
print(data["response"])

print("\nModel Information:")
print("Model:", data.get("model"))
print("Prompt tokens:", data.get("prompt_eval_count"))
print("Generated tokens:", data.get("eval_count"))

Generated Text:
Artificial intelligence (AI) refers to the development of computer systems that can perform tasks that would typically require human intelligence, such as:

1. Learning: AI systems can learn from data and improve their performance over time.
2. Reasoning: AI systems can draw conclusions based on available information and make decisions.
3. Problem-solving: AI systems can identify problems and develop solutions.

AI is a broad field that encompasses various subfields, including:

1. Machine learning (ML): A type of AI that enables machines to learn from data without being explicitly programmed.
2. Natural language processing (NLP): A type of AI that enables computers to understand, interpret, and generate human-like text or speech.
3. Computer vision: A type of AI that enables computers to interpret and understand visual information from images and videos.

AI systems can be categorized into three types based on their level of autonomy:

1. Narrow or weak AI: These syste

## Step 5: Inspect Model Configuration

Display the local model configuration to verify the model architecture and quantization information provided by Ollama.

In [9]:
import subprocess

result = subprocess.run(
    ["ollama", "show", "llama3:8b"],
    capture_output=True,
    text=True
)

print(result.stdout)

  Model
    architecture        llama    
    parameters          8.0B     
    context length      8192     
    embedding length    4096     
    quantization        Q4_0     

  Capabilities
    completion    

  Parameters
    num_keep    24                       
    stop        "<|start_header_id|>"    
    stop        "<|end_header_id|>"      
    stop        "<|eot_id|>"             

  License
    META LLAMA 3 COMMUNITY LICENSE AGREEMENT             
    Meta Llama 3 Version Release Date: April 18, 2024    
    ...                                                  




## Conclusion

The local Llama model analysis pipeline was implemented successfully. The workflow demonstrates model loading, quantization concepts, text generation, token logit extraction, hidden-state inspection, attention analysis, and dynamic entropy calculation across generation steps.